In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Install dependencies

In [2]:
!pip install -q bitsandbytes>=0.46.1 transformers>=4.45.0 peft trl datasets accelerate huggingface_hub

HF Login & Setup

In [3]:
from huggingface_hub import login
login(token="YOUR_HF_TOKEN")

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    total_mem = torch.cuda.mem_get_info(0)[1] / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} ({total_mem:.1f} GB)")

CUDA available: True
GPU: Tesla T4 (15.6 GB)


Verify data

In [14]:
import json
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/symptom")
train_path = DATA_DIR / "symptom_classifier_train.jsonl"
test_path  = DATA_DIR / "symptom_classifier_test.jsonl"

for p in [train_path, test_path]:
    with open(p) as f:
        lines = f.readlines()
    print(f"{p.name}: {len(lines)} samples")
    sample = json.loads(lines[0])
    print(f"  keys: {list(sample.keys())}")
    print()

symptom_classifier_train.jsonl: 15830 samples
  keys: ['instruction', 'input', 'output', 'disease', 'department', 'urgency']

symptom_classifier_test.jsonl: 738 samples
  keys: ['instruction', 'input', 'output', 'disease', 'department', 'urgency']



Load model with QLoRA

In [15]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

trainable params: 2,293,760 || all params: 3,215,043,584 || trainable%: 0.0713


Prepare dataset

In [6]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": str(train_path),
    "test":  str(test_path),
})
print(dataset)

SYSTEM_PROMPT = (
    "You are a medical triage assistant. Based on the patient's symptoms, "
    "classify them into the correct medical department and urgency level.\n"
    "Respond ONLY in this format:\n"
    "Department: <department name>\n"
    "Urgency: <Routine|Urgent|Emergency>"
)

def format_chat(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": example["input"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

train_ds = dataset["train"].map(format_chat, remove_columns=dataset["train"].column_names)
test_ds  = dataset["test"].map(format_chat, remove_columns=dataset["test"].column_names)

print(f"Train: {len(train_ds)} | Test: {len(test_ds)}")
print(f"\nSample:\n{train_ds[0]['text'][:500]}...")

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'disease', 'department', 'urgency'],
        num_rows: 15830
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'disease', 'department', 'urgency'],
        num_rows: 738
    })
})


Map:   0%|          | 0/15830 [00:00<?, ? examples/s]

Map:   0%|          | 0/738 [00:00<?, ? examples/s]

Train: 15830 | Test: 738

Sample:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 13 Mar 2026

You are a medical triage assistant. Based on the patient's symptoms, classify them into the correct medical department and urgency level.
Respond ONLY in this format:
Department: <department name>
Urgency: <Routine|Urgent|Emergency><|eot_id|><|start_header_id|>user<|end_header_id|>

Patient reports: burning micturition, bladder discomfort, foul smell of urine.<|eot_id|><|st...


Train

In [17]:

OUTPUT_DIR = "/content/drive/MyDrive/symptom"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    bf16=True,
    logging_steps=50,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
)


trainer = SFTTrainer(
  model=model,
  processing_class=tokenizer,
  train_dataset=train_ds,
  eval_dataset=test_ds,
  args=sft_config,
)

print(f"Training samples: {len(train_ds)}")
print(f"Steps/epoch: {len(train_ds) // (4 * 2)}")
print(f"Total steps: {len(train_ds) // (4 * 2) * 3}")
print("Starting training...")

trainer.train()

Training samples: 15830
Steps/epoch: 1978
Total steps: 5934
Starting training...


Epoch,Training Loss,Validation Loss
1,0.055910,0.063058
2,0.052617,0.057728
3,0.051474,0.056789


TrainOutput(global_step=5937, training_loss=0.09522512835727028, metrics={'train_runtime': 13530.2113, 'train_samples_per_second': 3.51, 'train_steps_per_second': 0.439, 'total_flos': 1.1166461743806874e+17, 'train_loss': 0.09522512835727028})

In [20]:
FINAL_ADAPTER = "/content/drive/MyDrive/medi-agent/data/processed/symptom/final_adapter"
trainer.model.save_pretrained(FINAL_ADAPTER)
tokenizer.save_pretrained(FINAL_ADAPTER)
print(f"Adapter saved to {FINAL_ADAPTER}")

Adapter saved to /content/drive/MyDrive/medi-agent/data/processed/symptom/final_adapter
